# 07. Many tests

When you test **many** effects or experiments, the chance that a false positive shows up by accident grows. Multiple-testing correction adjusts the p-values to control that risk. `ExperimentComparison` does it when comparing independent experiments.

## Four independent experiments

One has a true effect (`feature_A`); the other three are null. Each experiment is a CRD estimated with `NeymanCI`, which produces a p-value.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.inference import NeymanCI


def run_experiment(true_ate, seed, n=120):
    rng = np.random.default_rng(seed)
    y0 = rng.normal(size=n)
    y1 = y0 + true_ate
    df = pd.DataFrame({"x": rng.normal(size=n)})
    design = CRD(p=0.5, seed=seed)
    a = design.randomize(df)
    t = a.data_[a.treatment_col_].to_numpy()
    data = a.data_.copy()
    data["y"] = np.where(t == 1, y1, y0)
    a = CRDAssignment(data=data, treatment_col=a.treatment_col_, design=design, seed=seed)
    return NeymanCI(estimator=DifferenceInMeans(outcome_col="y")).fit(a).estimate()


experiments = {
    "feature_A": run_experiment(0.7, seed=1),   # true effect
    "feature_B": run_experiment(0.0, seed=2),   # null
    "feature_C": run_experiment(0.0, seed=3),   # null
    "feature_D": run_experiment(0.0, seed=4),   # null
}
{name: round(res.p_value, 4) for name, res in experiments.items()}

## Compare with correction (Holm)

`ExperimentComparison` collects the p-values and applies the correction across the whole family. The `p_value_corrected` column is what decides significance.

In [ ]:
from skxperiments.pipeline import ExperimentComparison

comp = ExperimentComparison(correction="holm", alpha=0.05).run(experiments)
comp.table[["experiment", "ate", "p_value", "p_value_corrected", "significant"]]

## FWER vs. FDR

- **Bonferroni / Holm** control the **FWER**: the chance of **any** false positive in the family. More conservative.
- **Benjamini-Hochberg (BH)** controls the **FDR**: the **proportion** of false positives among discoveries. More power when there are many tests.

The choice depends on the cost of a false positive in your context.

In [ ]:
comp_holm = ExperimentComparison(correction="holm").run(experiments)
comp_bh = ExperimentComparison(correction="bh").run(experiments)
print("Significant (Holm):", comp_holm.significant)
print("Significant (BH):  ", comp_bh.significant)

## What we learned

- Testing many things inflates false positives; correction adjusts the p-values to contain it.
- FWER (Holm/Bonferroni) is stricter; FDR (BH) has more power when there are many tests.
- `ExperimentComparison` applies the correction and returns a table ready for the forest plot.

**Next:** `08. Trust your experiment` covers the diagnostics (SRM, A/A, balance).